# Panduan Pelatihan Model mT5-small untuk Sistem Tanya Jawab Bahasa Indonesia (TutorQA)

Notebook ini dirancang untuk melatih model **google/mt5-small** pada dataset **TyDi QA** (Bahasa Indonesia). Proses pelatihan disesuaikan untuk dijalankan di Google Colab menggunakan runtime GPU T4 dengan mengutamakan kestabilan numerik dan efisiensi memori.

### Arsitektur Pelatihan:
1. **Adafactor Optimizer**: Digunakan sebagai pengganti AdamW karena model T5/mT5 sangat sensitif terhadap akumulasi gradien dan sering mengalami loss NaN (numerical instability) selama pelatihan. Adafactor menjaga kestabilan skala pembaruan parameter dan lebih hemat VRAM.
2. **Gradient Accumulation (Akumulasi Gradien)**: Mensimulasikan batch size efektif sebesar 16 secara virtual dengan melakukan akumulasi langkah gradien (accumulation steps = 4) pada batch fisik kecil (batch size = 4). Hal ini mencegah resiko kehabisan memori GPU (Out-Of-Memory/OOM).
3. **Pemberhentian Awal (Early Stopping)**: Menghentikan pelatihan secara otomatis apabila loss evaluasi tidak menunjukkan perbaikan dalam beberapa epoch berturut-turut untuk menghindari overfitting.
4. **Metrik Evaluasi ROUGE**: Digunakan untuk mengukur kemiripan tekstual jawaban yang dihasilkan oleh model dengan jawaban referensi secara kuantitatif selama tahapan evaluasi.
5. **Dynamic Padding**: Mengabaikan padding statis pada proses awal tokenisasi, sehingga ukuran padding disesuaikan secara dinamis untuk setiap batch selama collation guna mempercepat proses pelatihan.
6. **Indonesian QA Alignment**: Format prompt yang digunakan diselaraskan dengan format penyajian dari aplikasi backend TutorQA (`pertanyaan: ... konteks: ...`).

In [1]:
# Sel 1: Pemasangan library yang diperlukan
!pip install --upgrade transformers datasets evaluate accelerate sentencepiece rouge-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00


## 1. Memuat dan Mempersiapkan Dataset TyDi QA (Bahasa Indonesia)
Dataset TyDi QA dibatasi pada data sekunder untuk bahasa Indonesia.

In [2]:
# Sel 2: Unduh dan saring data bahasa Indonesia
from datasets import load_dataset

print("Mengunduh dataset TyDi QA...")
dataset = load_dataset("tydiqa", "secondary_task")

print("Menyaring data Bahasa Indonesia...")
ds_id_train = dataset['train'].filter(lambda x: x['id'].startswith('indonesian'))
ds_id_val = dataset['validation'].filter(lambda x: x['id'].startswith('indonesian'))

print(f"Jumlah data latih (train): {len(ds_id_train)}")
print(f"Jumlah data validasi (validation): {len(ds_id_val)}")

Mengunduh dataset TyDi QA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

secondary_task/train-00000-of-00001.parq(…):   0%|          | 0.00/26.9M [00:00<?, ?B/s]

secondary_task/validation-00000-of-00001(…):   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/5077 [00:00<?, ? examples/s]

Menyaring data Bahasa Indonesia...


Filter:   0%|          | 0/49881 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5077 [00:00<?, ? examples/s]

Jumlah data latih (train): 5702
Jumlah data validasi (validation): 565


## 2. Memuat Model dan Tokenizer mT5-small
Tokenizer dan model dimuat dari model dasar google/mt5-small.

In [3]:
# Sel 3: Memuat Tokenizer dan Model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "google/mt5-small"

print(f"Memuat tokenizer dan model dari {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

Memuat tokenizer dan model dari google/mt5-small...


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 3. Preprocessing dan Tokenisasi Dataset
Pada tahap ini, pertanyaan dan konteks digabungkan dengan format prompt terstruktur: `pertanyaan: {soal} konteks: {materi}`. Data dipotong (truncation) sesuai kapasitas model, sedangkan padding akan ditangani secara dinamis oleh collator.

In [4]:
# Sel 4: Fungsi Preprocessing
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = ["pertanyaan: " + q + " konteks: " + c for q, c in zip(examples["question"], examples["context"])]
    targets = [answers["text"][0] if len(answers["text"]) > 0 else "" for answers in examples["answers"]]

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Memulai pemrosesan dataset...")
tokenized_train = ds_id_train.map(preprocess_function, batched=True, remove_columns=ds_id_train.column_names)
tokenized_val = ds_id_val.map(preprocess_function, batched=True, remove_columns=ds_id_val.column_names)
print("Pemrosesan selesai.")

Memulai pemrosesan dataset...


Map:   0%|          | 0/5702 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Pemrosesan selesai.


## 4. Konfigurasi Pengukuran Kinerja Model (Metrik ROUGE)
Fungsi evaluasi disiapkan untuk menghitung skor ROUGE secara berkala guna melacak perkembangan akurasi teks jawaban hasil generate model terhadap jawaban referensi asli.

In [5]:
# Sel 5: Inisialisasi Metrik Evaluasi
import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {key: round(value * 100, 4) for key, value in result.items()}
    return result

## 5. Parameter Pelatihan dan Inisialisasi Trainer
Pelatihan dikonfigurasi menggunakan Adafactor optimizer untuk mencegah terjadinya NaN loss pada model berbasis T5. Parameter gradient accumulation disetel ke 4 langkah untuk menghasilkan kestabilan pembaruan bobot model.

In [6]:
# Sel 6: Setup Parameter Pelatihan
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="./hasil-qa-mt5",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    optim="adafactor",
    lr_scheduler_type="constant_with_warmup",
    warmup_ratio=0.1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=20,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 6. Proses Pelatihan Model (Fine-Tuning)
Jalankan sel ini untuk memulai proses pelatihan model.

In [7]:
# Sel 7: Menjalankan Pelatihan
print("Memulai pelatihan model...")
train_result = trainer.train()
print("Pelatihan selesai.")

Memulai pelatihan model...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,6.853734,0.971189,33.653400,23.696800,33.457200,33.502200
2,4.058903,0.700071,41.824000,28.954700,41.667800,41.656200
3,2.682602,0.622098,55.466400,40.486700,55.488900,55.427300
4,2.328137,0.638142,56.100900,41.411000,56.019000,56.041100
5,2.129350,0.611066,57.892100,43.773400,57.765500,57.911600


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1610: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Pelatihan selesai.


## 7. Pengujian Hasil Inferensi Model Secara Lokal
Lakukan uji coba tanya jawab langsung pada model yang telah selesai dilatih menggunakan contoh teks materi dan pertanyaan di bawah ini.

In [8]:
# Sel 8: Pengujian Inferensi
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

konteks_materi = """
Politeknik Caltex Riau (PCR) adalah perguruan tinggi swasta di Pekanbaru, Riau, Indonesia yang didirikan pada tahun 2001.
Kampus ini didirikan atas kerja sama antara Pemerintah Provinsi Riau dengan PT Caltex Pacific Indonesia.
PCR terkenal dengan program-program teknologi informasi, komputer, dan rekayasa tekniknya yang berstandar tinggi.
"""

pertanyaan = "Kapan Politeknik Caltex Riau didirikan?"

teks_input = f"pertanyaan: {pertanyaan} konteks: {konteks_materi}"

inputs = tokenizer(teks_input, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=64)

jawaban_ai = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*60)
print(f"Konteks: {konteks_materi.strip()}")
print(f"Pertanyaan: {pertanyaan}")
print(f"Jawaban AI: {jawaban_ai}")
print("="*60)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Konteks: Politeknik Caltex Riau (PCR) adalah perguruan tinggi swasta di Pekanbaru, Riau, Indonesia yang didirikan pada tahun 2001.
Kampus ini didirikan atas kerja sama antara Pemerintah Provinsi Riau dengan PT Caltex Pacific Indonesia.
PCR terkenal dengan program-program teknologi informasi, komputer, dan rekayasa tekniknya yang berstandar tinggi.
Pertanyaan: Kapan Politeknik Caltex Riau didirikan?
Jawaban AI: 2001


## 8. Menghubungkan Google Drive & Menyimpan Model
Hubungkan notebook dengan Google Drive Anda untuk menyimpan bobot model dan tokenizer hasil latihan agar dapat diunduh dan digunakan kembali secara lokal.

In [9]:
# Sel 9: Penyimpanan Model ke Google Drive
from google.colab import drive
import os

print("Menghubungkan ke Google Drive...")
drive.mount('/content/drive')

path_simpan = '/content/drive/MyDrive/NLP/mt5-qa-indonesia'

if not os.path.exists(path_simpan):
    os.makedirs(path_simpan)

print(f"Menyimpan model ke folder Google Drive: {path_simpan}...")
trainer.save_model(path_simpan)
tokenizer.save_pretrained(path_simpan)

print("Model dan tokenizer telah disimpan di Google Drive Anda.")

Menghubungkan ke Google Drive...
Mounted at /content/drive
Menyimpan model ke folder Google Drive: /content/drive/MyDrive/NLP/mt5-qa-indonesia...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model dan tokenizer telah disimpan di Google Drive Anda.
